In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb

In [3]:
train = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
test = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")

y = train["isFraud"]
test_ids = test["TransactionID"]

X = train.drop(columns=["isFraud"])

In [4]:
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(X[num_cols].median())

X[cat_cols] = X[cat_cols].fillna("missing")
test[cat_cols] = test[cat_cols].fillna("missing")


In [5]:
for df in [X, test]:

    df["log_amt"] = np.log1p(df["TransactionAmt"])
    df["null_count"] = df.isnull().sum(axis=1)
    

/tmp/ipykernel_57/3044457581.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["log_amt"] = np.log1p(df["TransactionAmt"])
/tmp/ipykernel_57/3044457581.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["null_count"] = df.isnull().sum(axis=1)
/tmp/ipykernel_57/3044457581.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, 

In [6]:
for col in cat_cols:

    freq = X[col].value_counts(normalize=True)

    X[col] = X[col].map(freq).fillna(0)
    test[col] = test[col].map(freq).fillna(0)

In [8]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

In [9]:
model = lgb.LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.02,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=0.2,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

In [10]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(test))
scores = []

X = X.fillna(0)
test = test.fillna(0)

for fold, (tr, val) in enumerate(kf.split(X, y)):

    print(f"Fold {fold+1}")

    X_train, X_val = X.iloc[tr], X.iloc[val]
    y_train, y_val = y.iloc[tr], y.iloc[val]

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )

    val_pred = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_pred)

    print("AUC:", auc)
    scores.append(auc)

    test_preds += model.predict_proba(test)[:, 1] / 5

Fold 1
[LightGBM] [Info] Number of positive: 16531, number of negative: 455901
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.593116 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37035
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 392
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034991 -> initscore=-3.317038
[LightGBM] [Info] Start training from score -3.317038
AUC: 0.8878547170916437
Fold 2
[LightGBM] [Info] Number of positive: 16531, number of negative: 455901
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.594274 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37019
[LightGBM] [Info] Number of data points in the train set: 472432, n

In [11]:
print("CV AUC:", np.mean(scores))

CV AUC: 0.88791704358929


In [12]:
submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": test_preds
})

submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")

Saved submission.csv


In [15]:
!pip install mlflow dagshub -q

In [16]:
import mlflow
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "lkhar21"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "d2982c921140413d4212c1e34543c62d25a3c42c"

mlflow.set_tracking_uri("https://dagshub.com/lkhar21/ML-assignment2.mlflow")

mlflow.set_experiment("LIGHTGBM_FRAUD")

2026/05/07 19:02:30 INFO mlflow.tracking.fluent: Experiment with name 'LIGHTGBM_FRAUD' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/50520c78309e4daeaac66d285a0e5eb5', creation_time=1778180550722, experiment_id='6', last_update_time=1778180550722, lifecycle_stage='active', name='LIGHTGBM_FRAUD', tags={}, trace_location=None, workspace='default'>

In [17]:
import mlflow
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

with mlflow.start_run(run_name="LightGBM_Fraud"):

    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict_proba(X_val)[:, 1]
    pred_class = (preds > 0.5).astype(int)

    auc = roc_auc_score(y_val, preds)
    f1 = f1_score(y_val, pred_class)
    prec = precision_score(y_val, pred_class)
    rec = recall_score(y_val, pred_class)

   
    mlflow.log_param("model", "LightGBM")
    mlflow.log_param("n_estimators", 2000)
    mlflow.log_param("learning_rate", 0.02)

    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)

    mlflow.sklearn.log_model(model, "model")

    print("AUC:", auc)

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.603181 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37112
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 392
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


2026/05/07 19:05:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 19:05:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


AUC: 0.9641605570686073
🏃 View run LightGBM_Fraud at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/6/runs/7198c86db6cd4022bb38e4f047e5d9f9
🧪 View experiment at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/6
